In [1]:
import json
from collections import defaultdict, Counter
import statistics
import pandas as pd

single_path = "../../../results/phase1_easy/single/details.jsonl"
repair_path = "../../../results/phase1_easy/repair/details.jsonl"
planner_path = "../../../results/phase1_easy/planner_coder/details.jsonl"

def load_jsonl(path):
    rows = []
    with open(path, "r") as f:
        for line in f:
            rows.append(json.loads(line))
    return rows

single_rows = load_jsonl(single_path)
repair_rows = load_jsonl(repair_path)
planner_rows = load_jsonl(planner_path)

len(single_rows), len(repair_rows), len(planner_rows)

(164, 245, 164)

In [2]:
# repair를 task-level final row로 압축
def group_by_task(rows):
    grouped = defaultdict(list)
    for r in rows:
        grouped[r["task_id"]].append(r)
    for task_id in grouped:
        grouped[task_id] = sorted(grouped[task_id], key=lambda x: x["attempt_idx"])
    return grouped

repair_grouped = group_by_task(repair_rows)

repair_final_rows = []
for task_id, attempts in repair_grouped.items():
    first = attempts[0]
    last = attempts[-1]

    solved_on_attempt = None
    for row in attempts:
        if row["status"] == "pass":
            solved_on_attempt = row["attempt_idx"]
            break

    repair_final_rows.append({
        "task_id": task_id,
        "method": "repair",
        "initial_status": first["status"],
        "initial_error_type": first["error_type"],
        "final_status": last["status"],
        "final_error_type": last["error_type"],
        "passed": last["status"] == "pass",
        "timeout": last["status"] == "timeout",
        "solved_on_attempt": solved_on_attempt,
        "num_attempts": len(attempts),
        "latency_sec_sum": sum(
            x["latency_sec"] for x in attempts if x.get("latency_sec") is not None
        ),
    })

len(repair_final_rows)

164

In [3]:
# single / planner도 비교용 포맷으로 맞추기
single_final_rows = []
for r in single_rows:
    single_final_rows.append({
        "task_id": r["task_id"],
        "method": "single",
        "initial_status": r["status"],
        "initial_error_type": r["error_type"],
        "final_status": r["status"],
        "final_error_type": r["error_type"],
        "passed": r["status"] == "pass",
        "timeout": r["status"] == "timeout",
        "solved_on_attempt": 0 if r["status"] == "pass" else None,
        "num_attempts": 1,
        "latency_sec_sum": r.get("latency_sec"),
    })

planner_final_rows = []
for r in planner_rows:
    planner_final_rows.append({
        "task_id": r["task_id"],
        "method": "planner_coder",
        "initial_status": r["status"],
        "initial_error_type": r["error_type"],
        "final_status": r["status"],
        "final_error_type": r["error_type"],
        "passed": r["status"] == "pass",
        "timeout": r["status"] == "timeout",
        "solved_on_attempt": 0 if r["status"] == "pass" else None,
        "num_attempts": 1,
        "latency_sec_sum": r.get("latency_sec"),
    })

len(single_final_rows), len(planner_final_rows)

(164, 164)

In [4]:
compare_df = pd.DataFrame(single_final_rows + repair_final_rows + planner_final_rows)
compare_df.head()

summary_df = (
    compare_df.groupby("method")
    .agg(
        total=("task_id", "count"),
        passed=("passed", "sum"),
        timeout=("timeout", "sum"),
        avg_latency=("latency_sec_sum", "mean"),
    )
    .reset_index()
)

summary_df["failed"] = summary_df["total"] - summary_df["passed"] - summary_df["timeout"]
summary_df["success_rate"] = summary_df["passed"] / summary_df["total"]

summary_df = summary_df[
    ["method", "total", "passed", "failed", "timeout", "success_rate", "avg_latency"]
]
summary_df

,method,total,passed,failed,timeout,success_rate,avg_latency
0,planner_coder,164,134,30,0,0.817073,1.799176
1,repair,164,138,26,0,0.841463,3.389598
2,single,164,118,46,0,0.719512,2.117310


In [5]:
for _, row in summary_df.iterrows():
    print("=" * 60)
    print(f"Method: {row['method']}")
    print(f"Total: {row['total']}")
    print(f"Passed: {row['passed']}")
    print(f"Failed: {row['failed']}")
    print(f"Timeout: {row['timeout']}")
    print(f"Success Rate: {row['success_rate']:.4f}")
    print(f"Avg Latency: {row['avg_latency']:.3f}s")

Method: planner_coder
Total: 164
Passed: 134
Failed: 30
Timeout: 0
Success Rate: 0.8171
Avg Latency: 1.799s
Method: repair
Total: 164
Passed: 138
Failed: 26
Timeout: 0
Success Rate: 0.8415
Avg Latency: 3.390s
Method: single
Total: 164
Passed: 118
Failed: 46
Timeout: 0
Success Rate: 0.7195
Avg Latency: 2.117s


In [6]:
# 에러 타입 비교
error_dist = (
    compare_df[compare_df["final_error_type"].notna()]
    .groupby(["method", "final_error_type"])
    .size()
    .reset_index(name="count")
    .sort_values(["method", "count"], ascending=[True, False])
)

error_dist

,method,final_error_type,count
0,planner_coder,assertion_error,18
3,planner_coder,syntax_error,6
1,planner_coder,name_error,4
2,planner_coder,runtime_error,1
4,planner_coder,value_error,1
5,repair,assertion_error,11
7,repair,syntax_error,11
6,repair,name_error,4
8,single,assertion_error,28
10,single,syntax_error,13


In [7]:
error_pivot = error_dist.pivot(
    index="final_error_type",
    columns="method",
    values="count"
).fillna(0).astype(int)

error_pivot

method,planner_coder,repair,single
final_error_type,,,
assertion_error,18,11,28
name_error,4,4,3
runtime_error,1,0,0
syntax_error,6,11,13
type_error,0,0,1
value_error,1,0,1


In [8]:
single_map = {r["task_id"]: r for r in single_final_rows}
repair_map = {r["task_id"]: r for r in repair_final_rows}
planner_map = {r["task_id"]: r for r in planner_final_rows}

repair_gain_cases = []
planner_gain_cases = []

for task_id in single_map:
    s = single_map[task_id]
    r = repair_map.get(task_id)
    p = planner_map.get(task_id)

    if r is not None and (not s["passed"]) and r["passed"]:
        repair_gain_cases.append(task_id)

    if p is not None and (not s["passed"]) and p["passed"]:
        planner_gain_cases.append(task_id)

len(repair_gain_cases), len(planner_gain_cases)
# repair gain 계산

(20, 29)

In [9]:
repair_only_gain = sorted(set(repair_gain_cases) - set(planner_gain_cases))
planner_only_gain = sorted(set(planner_gain_cases) - set(repair_gain_cases))
both_gain = sorted(set(repair_gain_cases) & set(planner_gain_cases))

len(repair_only_gain), len(planner_only_gain), len(both_gain)

(2, 11, 18)

In [10]:
# 어떤 task를 repair만 고쳤는지 / planner만 고쳤는지
repair_only_gain[:10], planner_only_gain[:10], both_gain[:10]

(['HumanEval/111', 'HumanEval/2'],
 ['HumanEval/100',
  'HumanEval/129',
  'HumanEval/139',
  'HumanEval/141',
  'HumanEval/37',
  'HumanEval/43',
  'HumanEval/62',
  'HumanEval/64',
  'HumanEval/76',
  'HumanEval/91'],
 ['HumanEval/105',
  'HumanEval/110',
  'HumanEval/114',
  'HumanEval/121',
  'HumanEval/137',
  'HumanEval/147',
  'HumanEval/149',
  'HumanEval/155',
  'HumanEval/159',
  'HumanEval/27'])

In [11]:
# single 실패 중 repair/planner가 어떻게 바꿨는지
rows = []
for task_id in single_map:
    s = single_map[task_id]
    r = repair_map.get(task_id)
    p = planner_map.get(task_id)

    rows.append({
        "task_id": task_id,
        "single_status": s["final_status"],
        "single_error": s["final_error_type"],
        "repair_status": r["final_status"] if r else None,
        "repair_error": r["final_error_type"] if r else None,
        "planner_status": p["final_status"] if p else None,
        "planner_error": p["final_error_type"] if p else None,
    })

transition_df = pd.DataFrame(rows)
transition_df.head()

,task_id,single_status,single_error,repair_status,repair_error,planner_status,planner_error
0,HumanEval/0,pass,None,pass,None,pass,None
1,HumanEval/1,pass,None,pass,None,pass,None
2,HumanEval/2,fail,assertion_error,pass,None,fail,syntax_error
3,HumanEval/3,pass,None,pass,None,pass,None
4,HumanEval/4,pass,None,pass,None,pass,None


In [12]:
# single에서 실패한 케이스만
failed_from_single = transition_df[transition_df["single_status"] != "pass"]
failed_from_single.head(20)

,task_id,single_status,single_error,repair_status,repair_error,planner_status,planner_error
2,HumanEval/2,fail,assertion_error,pass,None,fail,syntax_error
10,HumanEval/10,fail,assertion_error,fail,name_error,fail,name_error
26,HumanEval/26,fail,assertion_error,fail,assertion_error,fail,assertion_error
27,HumanEval/27,fail,syntax_error,pass,None,pass,None
33,HumanEval/33,fail,assertion_error,pass,None,pass,None
37,HumanEval/37,fail,assertion_error,fail,syntax_error,pass,None
40,HumanEval/40,fail,syntax_error,pass,None,pass,None
43,HumanEval/43,fail,syntax_error,fail,syntax_error,pass,None
46,HumanEval/46,fail,syntax_error,pass,None,pass,None
50,HumanEval/50,fail,syntax_error,fail,syntax_error,fail,name_error


In [13]:
comparison_rows = []

for task_id in single_map:
    s = single_map[task_id]["passed"]
    r = repair_map.get(task_id, {}).get("passed", False)
    p = planner_map.get(task_id, {}).get("passed", False)

    label = None
    if r and p:
        label = "both_success"
    elif r and not p:
        label = "repair_only_success"
    elif p and not r:
        label = "planner_only_success"
    else:
        label = "both_fail_or_single_only"

    comparison_rows.append({
        "task_id": task_id,
        "label": label
    })

comparison_label_df = pd.DataFrame(comparison_rows)
comparison_label_df["label"].value_counts()
# repair vs planner 누가 더 잘했는지

label
both_success                123
repair_only_success          15
both_fail_or_single_only     15
planner_only_success         11
Name: count, dtype: int64

In [14]:
# repair가 어떤 초기 에러를 잘 고쳤는지
repair_initial_error_df = pd.DataFrame(repair_final_rows)

repair_initial_error_df.groupby(
    ["initial_error_type", "passed"]
).size().reset_index(name="count").sort_values(
    ["initial_error_type", "passed", "count"],
    ascending=[True, False, False]
)

,initial_error_type,passed,count
1,assertion_error,True,7
0,assertion_error,False,21
3,name_error,True,1
2,name_error,False,2
5,syntax_error,True,11
4,syntax_error,False,2
6,type_error,False,1
7,value_error,True,1


In [15]:
planner_compare_df = pd.DataFrame(planner_final_rows)
planner_compare_df["final_error_type"].value_counts(dropna=False)
# planner가 어떤 final에러를 만드는지

final_error_type
None               134
assertion_error     18
syntax_error         6
name_error           4
runtime_error        1
value_error          1
Name: count, dtype: int64

In [16]:
latency_compare = compare_df.groupby("method").agg(
    avg_latency=("latency_sec_sum", "mean"),
    median_latency=("latency_sec_sum", "median"),
    max_latency=("latency_sec_sum", "max"),
).reset_index()

latency_compare

,method,avg_latency,median_latency,max_latency
0,planner_coder,1.799176,1.667455,4.879413
1,repair,3.389598,2.375491,18.375241
2,single,2.117310,1.793190,6.614124


In [17]:
task_id = "HumanEval/2"

print("=" * 100)
print("SINGLE RAW")
for r in single_rows:
    if r["task_id"] == task_id:
        print(r)
        break

print("=" * 100)
print("REPAIR RAW")
for r in repair_rows:
    if r["task_id"] == task_id:
        print(r)
        print("-" * 40)

print("=" * 100)
print("PLANNER RAW")
for r in planner_rows:
    if r["task_id"] == task_id:
        print(r)
        break

SINGLE RAW
{'dataset': 'humaneval', 'task_id': 'HumanEval/2', 'entry_point': 'truncate_number', 'method': 'single', 'model_name': 'Qwen/Qwen2.5-Coder-7B-Instruct', 'attempt_idx': 0, 'status': 'fail', 'passed': False, 'timeout': False, 'raw_output': '    return number - int(number)\n\n# Test cases to validate the correctness of the function\nassert truncate_number(3.5) == 0.5\nassert truncate_number(4.75) == 0.75\nassert truncate_number(9.99) == 0.99\nassert truncate_number(0.123) == 0.123\nassert truncate_number(10.0) == 0.0\n```', 'generated_code': '\n\ndef truncate_number(number: float) -> float:\n    """ Given a positive floating point number, it can be decomposed into\n    and integer part (largest integer smaller than given number) and decimals\n    (leftover part always smaller than 1).\n\n    Return the decimal part of the number.\n    >>> truncate_number(3.5)\n    0.5\n    """\n    return number - int(number)\n\n# Test cases to validate the correctness of the function\nassert t